# E6 — binning déterministe : ce qui a fait tomber une graine à 10 s> **Notebook v1, compilé le 2026-08-17.** La cellule 1 réaffiche ce numéro.**Prérequis : E5 doit avoir tourné**, puisque c'est lui qui a produit les graines 4 et 5à 10 s auxquelles on se compare.## La questionUne graine sur cinq s'effondre à macro-F1 **0.8374** à l'intervalle de 10 s, les quatreautres restant au-dessus de 0.9999. Le manuscrit le rapportait comme « un échec isoléd'un seul ajustement » — ce qui **décrit** l'événement sans dire **pourquoi** unajustement échoue. Sur un benchmark, ce n'est pas assez.## Ce qui est déjà établi, hors GeNIS`experiments/e6/reproduce_binning.py` a montré trois choses :1. **Le bagging et le tirage de colonnes sont hors de cause.** La campagne fixe   `n_estimators`, `num_leaves` et `learning_rate`, et laisse le reste aux défauts :   `subsample=1.0`, `subsample_freq=0`, `colsample_bytree=1.0`. Aucun rééchantillonnage,   ni de lignes ni de colonnes.2. **Le profil est celui d'une défaillance côté entraînement**, pas d'un tirage de test   malheureux : dégradation ordonnée par la rareté des classes (Spearman 0,93),   exactitude globale intacte à 0,9893, mais **ROC-AUC binaire tombé de 1,0000 à   0,9604** — le modèle n'ordonne plus — et **ajustement 15 % plus long** à budget   d'arbres fixe, donc une autre structure d'arbres.3. **Le tirage des bornes d'histogramme suffit.** À découpage fixe, sur une classe rare   à 0,23 % séparée par un intervalle étroit, seize graines dispersent son F1 en   **groupes discrets** ; en rendant le binning déterministe, les seize rendent la même   valeur au dernier chiffre.## Ce que ce notebook tranche, et qu'il est seul à pouvoir trancherDans ce papier, **une graine est la graine du découpage** : le détecteur est construit à`random_state=0` fixe. Changer de graine change donc **deux choses ensemble** :- la composition du jeu d'entraînement ;- la grille d'histogramme, que LightGBM pose à partir d'un tirage de 200 000 lignes  **de ce jeu d'entraînement**.Sur GeNIS elles sont confondues. Ce notebook les sépare en ne touchant qu'à la seconde.| Résultat | Conclusion ||---|---|| l'effondrement **disparaît** | c'est la grille d'histogramme || l'effondrement **persiste** | c'est la composition du découpage |**Les deux issues sont publiables.** Il n'y a pas de résultat décevant ici : dans un casle manuscrit nomme un mécanisme et le démontre, dans l'autre il élimine le candidat leplus vraisemblable et resserre la question sur la composition des découpages.## La couverture du tirage, qui est le cœur de l'affaire| intervalle | entraînement | couverture des bins ||---|---:|---:|| 5 s | 1 664 146 | 12,0 % || **10 s** | **883 279** | **22,6 %** || 30 s | 346 471 | 57,7 % || 60 s | 203 292 | **98,4 %** |À 60 s le tirage prend presque toutes les lignes : la grille y est quasi déterminée parles données, et c'est le seul intervalle où aucune graine ne s'écarte. La coïncidenceest ce qui rend l'hypothèse intéressante — elle ne la démontre pas.## Durée**Cinq ajustements LightGBM à 10 s**, environ 5 à 6 min chacun d'après les tempsenregistrés, plus le chargement des CSV. Compter **45 min à 1 h**. C'est sans communemesure avec E5. Le notebook est reprenable : il saute tout run déjà présent dans sonétat.**À me renvoyer :** `e6_results.json`.

In [ ]:
# --- 1. Drive, dossier, verrous ------------------------------------------E6_VERSION = "v1"; E6_BUILD = "2026-08-17"print(f"E6 notebook {E6_VERSION}, compile le {E6_BUILD}\n")import pathlib, json, sys, os, glob, shutil, time, gcfrom google.colab import drivedrive.mount("/content/drive")MYDRIVE = pathlib.Path("/content/drive/MyDrive")MARQUEUR = "article1_results.json"candidats = [MYDRIVE / "GeNIS" / "article1_final", MYDRIVE / "article1_final",             MYDRIVE / "GeNIS"]trouves = [c for c in candidats if (c / MARQUEUR).exists()]if not trouves:    for prof in ("*/", "*/*/", "*/*/*/"):        trouves = [p.parent for p in MYDRIVE.glob(prof + MARQUEUR)]        if trouves:            breakif not trouves:    sys.exit(f"{MARQUEUR} introuvable sous {MYDRIVE}.")SAVE = trouves[0]print(f"dossier de travail : {SAVE}")_R = json.loads((SAVE / MARQUEUR).read_text(encoding="utf-8"))# E5 est le prerequis : c'est lui qui a produit les graines 4 et 5 a 10 s.# Sans elles on ne compare que trois runs, ce qui est le point de depart du# probleme et pas son terme.E5 = SAVE / "e5_results.json"if not E5.exists():    sys.exit("e5_results.json introuvable : lance E5 d'abord. Sans les graines "             "4 et 5 a 10 s, la comparaison ne porte que sur trois runs.")_E5 = json.loads(E5.read_text(encoding="utf-8"))_T = _E5.get("temoin") or {}if not _T.get("reproduit"):    print("AVERTISSEMENT : le temoin de E5 n'avait pas reproduit la valeur\n"          "publiee. Les valeurs de reference ci-dessous sont celles de E5,\n"          "pas celles de la campagne, et c'est a elles qu'il faut comparer.\n")# Les cinq valeurs de reference a 10 s, graine par graine.REF = {}for s in (1, 2, 3):    REF[s] = _R["intervals"]["10"]["runs"][f"lightgbm|seed{s}"]["macro_f1"]for s in (4, 5):    k = f"10|lightgbm|seed{s}"    if k not in _E5["runs"]:        sys.exit(f"{k} absent de e5_results.json : E5 n'est pas complet a 10 s.")    REF[s] = _E5["runs"][k]["macro_f1"]print("\nreference a 10 s, binning echantillonne (l'existant) :")for s in sorted(REF):    print(f"   graine {s} : {REF[s]:.4f}" + ("   <-- l'effondrement" if REF[s] < .99 else ""))_EFF = [s for s in REF if REF[s] < .99]if not _EFF:    sys.exit("aucun effondrement dans la reference : rien a expliquer ici.")print(f"\ngraine(s) effondree(s) : {_EFF}")

In [ ]:
# --- 2. Corpus brut et pretraitement, identiques a E5 --------------------# Toute divergence ici rendrait la comparaison avec la reference nulle et non# avenue : c'est une recopie, pas une reecriture.import numpy as np, pandas as pdWORK = pathlib.Path("/content/genis"); WORK.mkdir(parents=True, exist_ok=True)os.chdir(WORK)drive_zip = MYDRIVE / "GeNIS" / "2-flows.zip"if not pathlib.Path("flows").exists():    if drive_zip.exists():        print("copie de 2-flows.zip depuis Drive…", flush=True)        shutil.copy(drive_zip, "2-flows.zip")    else:        print("telechargement depuis Zenodo (~380 Mo)…", flush=True)        os.system('wget -q "https://zenodo.org/records/14919237/files/'                  '2-flows.zip?download=1" -O 2-flows.zip')    os.system("unzip -o -q 2-flows.zip -d flows")csvs = sorted(glob.glob("flows/**/*.csv", recursive=True))assert csvs, "aucun CSV trouve"print(f"{len(csvs)} fichiers CSV")IDENT_LIST = ["FlowID", "AutoId", "SrcAddr", "DstAddr", "Ssaddr", "Sdaddr",              "SrcMac", "DstMac", "SrcOui", "DstOui", "Sport", "Dport",              "sIpId", "dIpId", "sMpls", "dMpls", "sAS", "dAS", "iAS",              "sCo", "dCo", "sVid", "dVid"]POS_LIST = ["StartTime", "LastTime", "Rank", "Seq"]LAB_LIST = ["BinaryLabel", "CategoryLabel", "SubCategoryLabel"]def feature_sets(d):    ident = [c for c in IDENT_LIST if c in d.columns]    pos = [c for c in POS_LIST if c in d.columns]    num = (d.drop(columns=ident + LAB_LIST, errors="ignore")             .select_dtypes(include=[np.number]).replace([np.inf, -np.inf], np.nan))    nu = num.nunique(dropna=True); const = nu[nu <= 1].index.tolist()    num = num.drop(columns=const).fillna(0.0).astype(np.float32)    return num, list(num.columns), [c for c in num.columns if c not in pos], pos, ident, constdef load_slice_df(iv):    files = [c for c in csvs if f"flows-{iv}-sec" in c]    d = pd.concat([pd.read_csv(c, low_memory=False) for c in files], ignore_index=True)    ysub = d["SubCategoryLabel"].astype(str).str.strip()    y9 = ysub.where(~ysub.str.startswith("benign"), "benign")    return d, y9BLACKLIST = _R["audit"]["blacklist"]CLASS_NAMES = _R["slice60"]["classes"]; C = len(CLASS_NAMES)BENIGN = CLASS_NAMES.index("benign")GRAINES = [1, 2, 3, 4, 5]print(f"\nliste noire : {len(BLACKLIST)} colonnes | classes : {C}")

In [ ]:
# --- 3. Le detecteur, et LE seul parametre qui change -------------------from sklearn.model_selection import train_test_splitfrom sklearn.preprocessing import RobustScalerfrom sklearn.metrics import f1_score, matthews_corrcoef, average_precision_score, roc_auc_scorefrom lightgbm import LGBMClassifier# Recopie exacte de la configuration publiee. subsample_for_bin est le SEUL# argument ajoute, et il n'est ajoute que dans le bras deterministe : c'est# ce qui permet d'attribuer une difference a lui et a rien d'autre.DECLARES = dict(n_estimators=300, num_leaves=63, learning_rate=.1)def make_lgbm(sub_bin=None):    kw = dict(DECLARES, n_jobs=2, random_state=0, verbose=-1)    if sub_bin is not None:        kw["subsample_for_bin"] = sub_bin    return LGBMClassifier(**kw)def evaluate(y_true, probs, fit_t, pred_t):    pred = probs.argmax(1); present = np.unique(y_true)    is_att, pred_att = y_true != BENIGN, pred != BENIGN    p_att = 1.0 - probs[:, BENIGN]    return {"accuracy": float((pred == y_true).mean()),            "macro_f1": float(f1_score(y_true, pred, labels=present, average="macro",                                       zero_division=0)),            "mcc": float(matthews_corrcoef(y_true, pred)),            "per_class_f1": {CLASS_NAMES[i]: float(v) for i, v in zip(range(C),                f1_score(y_true, pred, labels=range(C), average=None, zero_division=0))},            "binary": {"fpr": float(pred_att[~is_att].mean()) if (~is_att).any() else None,                       "roc_auc": float(roc_auc_score(is_att, p_att)) if 0 < is_att.mean() < 1 else None},            "fit_time_s": round(fit_t, 2), "predict_time_s": round(pred_t, 3),            "n_test": int(len(y_true))}def split_de(y_iv, s):    idx = np.arange(len(y_iv))    itr, itmp = train_test_split(idx, test_size=.4, random_state=s, stratify=y_iv)    iva_, ite_ = train_test_split(itmp, test_size=.5, random_state=s, stratify=y_iv[itmp])    return itr, iva_, ite_def un_run(s, Xiv, y_iv, sub_bin):    itr, _iva, ite_ = split_de(y_iv, s)    sc = RobustScaler().fit(Xiv[itr])    Xtr, Xte = [np.nan_to_num(sc.transform(Xiv[i_]), nan=0., posinf=0., neginf=0.)                .astype(np.float32) for i_ in (itr, ite_)]    ytr_, yte_ = y_iv[itr], y_iv[ite_]    mdl = make_lgbm(sub_bin)    t0 = time.time(); mdl.fit(Xtr, ytr_); ft = time.time() - t0    t0 = time.time(); pte = mdl.predict_proba(Xte); pt = time.time() - t0    out = evaluate(yte_, pte, ft, pt)    out["n_train"] = int(len(itr)); out["subsample_for_bin"] = sub_bin    del mdl, pte, Xtr, Xte; gc.collect()    return outSTATE_PATH = SAVE / "e6_results.json"def load_state():    if STATE_PATH.exists():        st = json.loads(STATE_PATH.read_text(encoding="utf-8"))        if st.get("meta", {}).get("version") == E6_VERSION:            return st        print("etat d'une autre version : on repart de zero.")    return {"meta": {"version": E6_VERSION, "build": E6_BUILD},            "reference": {str(k): v for k, v in REF.items()}, "runs": {}}def save_state(st):    tmp = STATE_PATH.with_suffix(".tmp")    tmp.write_text(json.dumps(st, indent=1, ensure_ascii=False, default=float),                   encoding="utf-8")    tmp.replace(STATE_PATH)STATE = load_state()print(f"etat : {len(STATE['runs'])} runs deja faits")

In [ ]:
# --- 4. Les cinq ajustements, binning rendu deterministe ----------------print("chargement de l'intervalle 10 s…", flush=True)d10, y9_10 = load_slice_df("10")num10, _, clean10, _, _, _ = feature_sets(d10)aud10 = [c for c in clean10 if c not in BLACKLIST]X10 = np.ascontiguousarray(num10[aud10].values, dtype=np.float32)y10 = np.array([CLASS_NAMES.index(v) for v in y9_10])del d10, num10; gc.collect()N_TR = int(round(0.6 * len(y10)))print(f"{len(y10):,} flux, {X10.shape[1]} colonnes auditees, "      f"entrainement ~{N_TR:,}")# Au-dela de la taille du jeu d'entrainement, LightGBM prend toutes les# lignes : la grille cesse de dependre d'un tirage.SUB = N_TR + 1print(f"subsample_for_bin = {SUB:,} (> {N_TR:,}) : binning deterministe\n")for s in GRAINES:    k = f"det|seed{s}"    if k in STATE["runs"]:        print(f"  graine {s} : deja faite ({STATE['runs'][k]['macro_f1']:.4f})")        continue    t0 = time.time()    STATE["runs"][k] = un_run(s, X10, y10, SUB)    save_state(STATE)    r = STATE["runs"][k]    print(f"  graine {s} : macro-F1 {r['macro_f1']:.4f}  "          f"(reference {REF[s]:.4f}, ecart {r['macro_f1']-REF[s]:+.4f})  "          f"[{time.time()-t0:.0f} s]", flush=True)# Un temoin, pour ne pas confondre "le binning repare" avec "cet# environnement ne reproduit rien". On rejoue une graine SAINE avec le# binning echantillonne : elle doit retomber sur sa valeur de reference._sain = [s for s in GRAINES if REF[s] >= .99][0]if "temoin|ech" not in STATE["runs"]:    print(f"\ntemoin : graine {_sain} (saine) avec le binning d'origine…", flush=True)    STATE["runs"]["temoin|ech"] = un_run(_sain, X10, y10, None)    STATE["runs"]["temoin|ech"]["graine"] = _sain    save_state(STATE)_t = STATE["runs"]["temoin|ech"]print(f"temoin graine {_sain} : {_t['macro_f1']:.4f} contre {REF[_sain]:.4f} publie, "      f"ecart {abs(_t['macro_f1']-REF[_sain]):.2e}")del X10, y10; gc.collect()

In [ ]:
# --- 5. Le verdict -------------------------------------------------------TOL = 1e-3det = {s: STATE["runs"][f"det|seed{s}"]["macro_f1"] for s in GRAINES}_t = STATE["runs"]["temoin|ech"]temoin_ok = abs(_t["macro_f1"] - REF[_t["graine"]]) <= TOLprint(f"{'graine':>8} {'echantillonne':>15} {'deterministe':>14} {'ecart':>9}")for s in GRAINES:    print(f"{s:>8} {REF[s]:>15.4f} {det[s]:>14.4f} {det[s]-REF[s]:>+9.4f}"          + ("   <-- effondrement d'origine" if REF[s] < .99 else ""))eff_avant = sorted(s for s in GRAINES if REF[s] < .99)eff_apres = sorted(s for s in GRAINES if det[s] < .99)print(f"\neffondrements avant : {eff_avant}")print(f"effondrements apres : {eff_apres}")STATE["verdict"] = {"temoin_reproduit": bool(temoin_ok),                    "effondrements_avant": eff_avant,                    "effondrements_apres": eff_apres,                    "deterministe": {str(k): v for k, v in det.items()}}if not temoin_ok:    STATE["verdict"]["conclusion"] = "indecidable"    print("\nINDECIDABLE. Le temoin ne reproduit pas sa valeur de reference, donc\n"          "une difference entre les deux colonnes peut venir de l'environnement\n"          "et pas du binning. Ne rien conclure : c'est le seul resultat honnete.")elif not eff_apres:    STATE["verdict"]["conclusion"] = "grille"    print("\nC'EST LA GRILLE D'HISTOGRAMME. Le binning rendu deterministe fait\n"          "disparaitre l'effondrement, toutes choses egales par ailleurs. La\n"          "section 6.4 peut nommer le mecanisme et le demontrer, et la\n"          "recommandation qui suit est concrete : aux intervalles courts, ou le\n"          "tirage de 200 000 lignes ne couvre qu'un cinquieme du jeu\n"          "d'entrainement, porter subsample_for_bin au-dela de sa taille.")elif eff_apres == eff_avant:    STATE["verdict"]["conclusion"] = "decoupage"    print("\nC'EST LA COMPOSITION DU DECOUPAGE. Le binning n'y est pour rien : le\n"          "candidat le plus vraisemblable est elimine et la question se resserre\n"          "sur ce que ce decoupage-la met a l'entrainement. Regarder d'abord les\n"          "flux bruteforce-ftp qu'il retient, dont le F1 tombe a 0.0561.")else:    STATE["verdict"]["conclusion"] = "mixte"    print("\nRESULTAT MIXTE : la liste des effondrements change sans disparaitre.\n"          "Le binning module le phenomene sans le causer seul. A rapporter tel\n"          "quel, avec les cinq valeurs.")save_state(STATE)print(f"\necrit : {STATE_PATH}")

## Ce qu'il faut me renvoyer`e6_results.json`, depuis le dossier de travail. Il contient :- `reference` — les cinq macro-F1 à 10 s tels qu'ils existent aujourd'hui ;- `runs` — les cinq ajustements à binning déterministe, au format complet  (macro-F1, F1 par classe, FPR, ROC-AUC, temps), **plus le témoin** ;- `verdict` — la conclusion, en clair.## Pourquoi il y a un témoinLe notebook rejoue aussi une graine **saine** avec le binning d'origine. Sans cecontrôle, une différence entre les deux colonnes serait ambiguë : elle pourrait venir dubinning, ou simplement du fait que cet environnement ne reproduit pas la campagne. Si letémoin ne retombe pas sur sa valeur publiée, **le notebook refuse de conclure**, et c'estle seul résultat honnête dans ce cas.## Les trois issues, et ce que chacune donne au manuscrit| Verdict | Ce que la §6.4 peut alors écrire ||---|---|| **grille** | le mécanisme est nommé et démontré, et la recommandation est concrète : aux intervalles courts, porter `subsample_for_bin` au-delà de la taille du jeu d'entraînement || **découpage** | le candidat le plus vraisemblable est éliminé ; la question se resserre sur ce que ce découpage met à l'entraînement, et l'aveu est plus net qu'aujourd'hui || **mixte** | le binning module sans causer seul ; à rapporter avec les cinq valeurs |Aucune de ces trois issues n'affaiblit le papier. La quatrième, *indécidable*, signifieseulement qu'il faut refaire tourner dans un environnement qui reproduit.## Ce que ce notebook ne fait pasIl ne retouche pas `article1_results.json` ni `e5_results.json`. Il n'exécute queLightGBM à 10 s : XGBoost est invariant sur ses vingt runs et le DNN décline pour uneraison qui n'a rien à voir, décrite en §6.4.